# 09.00 MAG7 - Visualização de Preços

Análise simples dos preços do MAG7: AAPL, MSFT, AMZN, GOOG, META, NVDA e TSLA.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from yfinance import download
import plotly.graph_objects as go

In [ ]:
mag7 = ["AAPL", "MSFT", "AMZN", "GOOG", "META", "NVDA", "TSLA"]

param = {
    'tickers': mag7,
    'period': '10y',
    'interval': '1mo',
    'auto_adjust': True,
}

raw = download(
    tickers=param['tickers'],
    period=param['period'],
    interval=param['interval'],
    auto_adjust=param['auto_adjust'],
    progress=False
)

if isinstance(raw.columns, pd.MultiIndex):
    if 'Close' in raw.columns.get_level_values(0):
        prices = raw['Close'].copy()
    elif 'Adj Close' in raw.columns.get_level_values(0):
        prices = raw['Adj Close'].copy()
    else:
        raise ValueError('Não foi possível encontrar colunas de fechamento no retorno do yfinance.')
else:
    prices = raw.copy()

prices = prices.dropna(how='all').ffill()
prices = prices[[ticker for ticker in mag7 if ticker in prices.columns]]

print(f"Tickers disponíveis: {', '.join(prices.columns)}")
print(f"Período: {prices.index.min().strftime('%d/%m/%Y')} até {prices.index.max().strftime('%d/%m/%Y')}")
prices.tail()

Tickers disponíveis: AAPL, MSFT, AMZN, GOOG, META, NVDA, TSLA
Período: 01/03/2016 até 01/02/2026


Ticker,AAPL,MSFT,AMZN,GOOG,META,NVDA,TSLA
Date,,,,,,,
2025-10-01,269.855652,515.665649,244.220001,281.636261,647.821655,202.478729,456.559998
2025-11-01,278.319519,489.972534,233.220001,319.911285,647.421997,176.990143,430.170013
2025-12-01,271.605835,482.518677,230.820007,313.595398,659.552124,186.489624,449.720001
2026-01-01,259.237427,429.310120,239.300003,338.529999,716.500000,191.130005,430.410004
2026-02-01,266.179993,384.470001,205.270004,311.690002,637.250000,191.550003,399.829987


In [ ]:
base100 = prices.div(prices.iloc[0]).mul(100)

fig = go.Figure()
for ticker in prices.columns:
    fig.add_trace(
        go.Scatter(
            x=prices.index,
            y=prices[ticker],
            mode='lines',
            name=ticker
        )
    )

fig.update_layout(
    title='MAG7 - Preços Históricos',
    xaxis_title='Data',
    yaxis_title='Preço (USD)',
    template='plotly_white',
    hovermode='x unified',
    legend_title='Ticker',
    height=550
)
fig.show()

fig_base = go.Figure()
for ticker in base100.columns:
    fig_base.add_trace(
        go.Scatter(
            x=base100.index,
            y=base100[ticker],
            mode='lines',
            name=ticker
        )
    )

fig_base.update_layout(
    title='MAG7 - Preços Normalizados (Base 100)',
    xaxis_title='Data',
    yaxis_title='Base 100',
    template='plotly_white',
    hovermode='x unified',
    legend_title='Ticker',
    height=550
)
fig_base.show()